# 🚀 Phase 3: NASA C-MAPSS 터보팬 엔진 예지보전 — LSTM 기반 RUL 예측

**프로젝트 목표:** 항공기 터보팬 엔진 센서 데이터의 시계열 열화(Degradation) 패턴을 LSTM 딥러닝으로 학습하여 고장까지 남은 잔존수명(RUL: Remaining Useful Life)을 정밀 예측합니다.

## 📋 파이프라인
1. 데이터 로드 (NASA C-MAPSS FD001)
2. RUL 라벨링 (Piecewise Linear + Capping @125 — 논문 표준)
3. 불필요 센서 제거 + Min-Max 정규화
4. Sliding Window → 3D Tensor 변환
5. 2-Layer LSTM + Dropout 모델 학습
6. RMSE / MAE / NASA Scoring Function 평가
7. RUL 예측 수렴 그래프 시각화

In [ ]:
# 필요 라이브러리 설치 및 임포트
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('✅ 환경 준비 완료')

## 1단계: 데이터 로드

In [ ]:
# NASA C-MAPSS FD001 데이터셋 로드
BASE_URL = 'https://raw.githubusercontent.com/CHADALAI/NASA-CMAPSS-Dataset/master/'
cols = ['id', 'cycle', 'set1', 'set2', 'set3'] + [f's{i}' for i in range(1, 22)]

train_df = pd.read_csv(BASE_URL + 'train_FD001.txt', sep=' ', header=None).dropna(axis=1)
train_df.columns = cols
test_df  = pd.read_csv(BASE_URL + 'test_FD001.txt',  sep=' ', header=None).dropna(axis=1)
test_df.columns = cols
y_test_df = pd.read_csv(BASE_URL + 'RUL_FD001.txt', sep=' ', header=None).dropna(axis=1)
y_test_df.columns = ['RUL']

print(f'Train: {train_df.shape} | Test: {test_df.shape} | Engines: {train_df.id.nunique()}')
train_df.head()

## 2단계: RUL 라벨링 (Piecewise Linear Capping — 논문 표준)

In [ ]:
# RUL = 해당 엔진 최대 사이클 - 현재 사이클
# Capping: RUL > 125이면 125로 고정 (Piecewise Linear — Heimes 2008 논문 표준)
RUL_CAP = 125

max_cycles = train_df.groupby('id')['cycle'].max().reset_index()
max_cycles.columns = ['id', 'max_cycle']
train_df = train_df.merge(max_cycles, on='id')
train_df['RUL'] = (train_df['max_cycle'] - train_df['cycle']).clip(upper=RUL_CAP)
train_df.drop('max_cycle', axis=1, inplace=True)

print(f'RUL 분포 (Capping @{RUL_CAP}):')
print(train_df['RUL'].describe())

# RUL 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train_df['RUL'].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('RUL 분포 (Capping @125 적용)')
axes[0].set_xlabel('RUL')
sample_engine = train_df[train_df['id'] == 1]
axes[1].plot(sample_engine['cycle'], sample_engine['RUL'], color='firebrick')
axes[1].set_title('엔진 #1 RUL 감소 곡선 (Piecewise Linear)')
axes[1].set_xlabel('사이클'); axes[1].set_ylabel('RUL')
plt.tight_layout(); plt.show()

## 3단계: 센서 선택 및 Min-Max 정규화

In [ ]:
# 상수 센서 제거 (표준 편차 == 0): s1, s5, s6, s10, s16, s18, s19
drop_sensors = ['s1', 's5', 's6', 's10', 's16', 's18', 's19']
drop_cols = ['set3'] + drop_sensors

feature_cols = [c for c in cols[2:] if c not in drop_cols]
print(f'사용 피처 ({len(feature_cols)}개):', feature_cols)

scaler = MinMaxScaler()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
test_df[feature_cols]  = scaler.transform(test_df[feature_cols])

## 4단계: Sliding Window → 3D Tensor 변환

In [ ]:
SEQ_LEN = 50  # 과거 50 사이클을 윈도우로 묶음

def create_sequences(df, feature_cols, seq_len, target='RUL'):
    X, y = [], []
    for engine_id in df['id'].unique():
        engine_df = df[df['id'] == engine_id].reset_index(drop=True)
        feat = engine_df[feature_cols].values
        label = engine_df[target].values
        for i in range(len(feat) - seq_len):
            X.append(feat[i:i+seq_len])
            y.append(label[i+seq_len])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

def create_test_sequences(df, feature_cols, seq_len):
    X = []
    for engine_id in df['id'].unique():
        engine_df = df[df['id'] == engine_id].reset_index(drop=True)
        feat = engine_df[feature_cols].values
        if len(feat) >= seq_len:
            X.append(feat[-seq_len:])
        else:
            pad = np.zeros((seq_len - len(feat), len(feature_cols)))
            X.append(np.vstack([pad, feat]))
    return np.array(X, dtype=np.float32)

X_train, y_train = create_sequences(train_df, feature_cols, SEQ_LEN)
X_test = create_test_sequences(test_df, feature_cols, SEQ_LEN)
y_test = y_test_df['RUL'].values.clip(max=RUL_CAP).astype(np.float32)

print(f'X_train: {X_train.shape} | y_train: {y_train.shape}')
print(f'X_test : {X_test.shape}  | y_test : {y_test.shape}')

## 5단계: LSTM 모델 설계 및 학습

In [ ]:
model = Sequential([
    LSTM(128, input_shape=(SEQ_LEN, len(feature_cols)), return_sequences=True),
    Dropout(0.2),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
]

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=256,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

## 6단계: 성능 평가 — RMSE / MAE / NASA Scoring Function

In [ ]:
y_pred = model.predict(X_test).flatten()

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)

# NASA Scoring Function: 늦게 예측하면 더 큰 페널티 부여 (안전 측면)
def nasa_score(y_true, y_pred):
    diff = y_pred - y_true
    score = np.where(diff < 0, np.exp(-diff/13) - 1, np.exp(diff/10) - 1)
    return np.sum(score)

score = nasa_score(y_test, y_pred)

print('=' * 40)
print(f'  RMSE  : {rmse:.4f}')
print(f'  MAE   : {mae:.4f}')
print(f'  NASA Score : {score:.2f}  (낮을수록 우수)')
print('=' * 40)

## 7단계: 시각화 — 학습 곡선 & RUL 예측 결과

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 2, figure=fig)

# 학습/검증 손실 곡선
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(history.history['loss'], label='Train Loss', color='steelblue')
ax1.plot(history.history['val_loss'], label='Val Loss', color='tomato', linestyle='--')
ax1.set_title('학습/검증 손실 (MSE)')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(alpha=0.3)

# 실제 vs 예측 RUL 산점도
ax2 = fig.add_subplot(gs[0, 1])
ax2.scatter(y_test, y_pred, alpha=0.5, color='steelblue', s=20)
lims = [0, RUL_CAP + 10]
ax2.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect Prediction')
ax2.set_title(f'실제 RUL vs 예측 RUL  (RMSE={rmse:.2f})')
ax2.set_xlabel('실제 RUL'); ax2.set_ylabel('예측 RUL')
ax2.legend(); ax2.grid(alpha=0.3)

# 오차 분포 히스토그램
ax3 = fig.add_subplot(gs[1, 0])
errors = y_pred - y_test
ax3.hist(errors, bins=40, color='mediumpurple', edgecolor='white')
ax3.axvline(0, color='red', linestyle='--')
ax3.set_title('예측 오차 분포 (Prediction Error Distribution)')
ax3.set_xlabel('예측 오차 (Predicted - True RUL)')
ax3.grid(alpha=0.3)

# 엔진별 예측 막대 비교 (첫 20개 엔진)
ax4 = fig.add_subplot(gs[1, 1])
idxs = range(20)
ax4.bar([i-0.2 for i in idxs], y_test[:20], width=0.4, label='실제 RUL', color='steelblue', alpha=0.8)
ax4.bar([i+0.2 for i in idxs], y_pred[:20], width=0.4, label='예측 RUL', color='tomato', alpha=0.8)
ax4.set_title('엔진별 RUL 비교 (처음 20개)')
ax4.set_xlabel('엔진 ID'); ax4.set_ylabel('RUL')
ax4.legend(); ax4.grid(axis='y', alpha=0.3)

plt.suptitle('NASA C-MAPSS FD001 LSTM RUL 예측 결과 종합', fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('CMAPSS_LSTM_Results.png', dpi=150, bbox_inches='tight')
plt.show()
print('결과 이미지 저장 완료: CMAPSS_LSTM_Results.png')